In [1]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import requests
import io
import re

log = load_log()
print(f"Log loaded. Rows: {len(log)}")

Log loaded. Rows: 5


## TI CPI Pipeline

**Source:** Transparency International Corruption Perceptions Index
**Access:** Automated direct download — no manual step required
**Download instructions:** See `docs/instructions_data_maintenance.md` — TI_CPI section

### Framework usage
| Indicator | Concept | Role |
|-----------|---------|------|
| CPI Score (0-100) | Control of corruption | Primary tier 1 |

In [2]:
import requests
import io
from datetime import datetime
import pandas as pd

# Our World in Data hosts TI CPI as a clean historical panel CSV
# Stable URL, no authentication required, updated annually
OWID_CPI_URL = "https://ourworldindata.org/grapher/ti-corruption-perception-index.csv?v=1&csvType=full&useColumnShortNames=false"

print("Downloading TI CPI from Our World in Data...")
response = requests.get(OWID_CPI_URL, timeout=30)
print(f"Status: {response.status_code}, Size: {len(response.content)/1024:.1f}KB")

# Load CSV
cpi_raw = pd.read_csv(io.StringIO(response.text))
print(f"\nShape: {cpi_raw.shape}")
print(f"Columns: {list(cpi_raw.columns)}")
print(f"Years: {sorted(cpi_raw['Year'].unique())}")
print(f"Countries: {cpi_raw['Code'].nunique()}")
print(cpi_raw.head(5))

Status: 200, Size: 65.8KB

Shape: (2312, 5)
Columns: ['Entity', 'Code', 'Year', 'Corruption Perceptions Index', 'World region according to OWID']
Years: [np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Countries: 182
        Entity Code  Year  Corruption Perceptions Index  \
0  Afghanistan  AFG  2012                             8   
1  Afghanistan  AFG  2013                             8   
2  Afghanistan  AFG  2014                            12   
3  Afghanistan  AFG  2015                            11   
4  Afghanistan  AFG  2016                            15   

  World region according to OWID  
0                           Asia  
1                           Asia  
2                           Asia  
3                           Asia  
4                           Asia  


In [3]:
# Filter and rename
cpi = cpi_raw.copy()
cpi = cpi.rename(columns={
    'Entity':                        'country_name',
    'Code':                          'country_code',
    'Year':                          'year',
    'Corruption Perceptions Index':  'ti_cpi_score',
})

# Drop OWID region column — not needed for framework
cpi = cpi.drop(columns=['World region according to OWID'])

# Filter to framework start year
cpi = cpi[cpi['year'] >= FRAMEWORK_START_YEAR].copy()

# Drop rows with no country code — these are regional aggregates added by OWID
cpi = cpi[cpi['country_code'].notna() & (cpi['country_code'] != '')].copy()

# Sort
cpi = cpi.sort_values(['country_name', 'year']).reset_index(drop=True)

print(f"Shape: {cpi.shape}")
print(f"Years: {sorted(cpi['year'].unique())}")
print(f"Countries: {cpi['country_name'].nunique()}")
print(f"\nMissing values (%):")
missing_pct = (cpi.isnull().sum() / len(cpi) * 100).round(1)
print(missing_pct[missing_pct > 0].sort_values(ascending=False))
print(cpi.head())

Shape: (2312, 4)
Years: [np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Countries: 182

Missing values (%):
Series([], dtype: float64)
  country_name country_code  year  ti_cpi_score
0  Afghanistan          AFG  2012             8
1  Afghanistan          AFG  2013             8
2  Afghanistan          AFG  2014            12
3  Afghanistan          AFG  2015            11
4  Afghanistan          AFG  2016            15


In [4]:
# Save to processed
output_path = os.path.join(PROCESSED_DIR, "ti_cpi_clean.csv")
cpi.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {cpi.shape}")

# Derive metadata from data — no hardcoding
latest_year = str(int(cpi['year'].max()))
data_as_of_date = latest_year

# Update download log
update_entry(
    "TI_CPI",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=data_as_of_date,
    local_filename="ti_cpi_clean.csv",
    latest_available_version=latest_year,
    notes=f"Downloaded from Our World in Data (OWID) historical panel. Source: TI CPI via OWID API. Covers 2012-present. TI direct files are password protected."
)

print_entry("TI_CPI")

Written: C:\Users\mjbou\governance-framework\data\processed\ti_cpi_clean.csv
Shape: (2312, 4)
[download_log] Updated entry for TI_CPI
  source_id: TI_CPI
  last_attempted_date: 2026-05-21
  last_successful_download_date: 2026-05-21
  data_as_of_date: 2024
  local_filename: ti_cpi_clean.csv
  latest_available_version: 2024
  no_update_reason: nan
  notes: Downloaded from Our World in Data (OWID) historical panel. Source: TI CPI via OWID API. Covers 2012-present. TI direct files are password protected.
